# 🚀 ShieldPay — 03: Model Training

Train and compare three models:
- **Logistic Regression** (baseline)
- **Random Forest** (ensemble)
- **XGBoost** (gradient boosting — our champion)

Then save the best model for the FastAPI backend.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, json, os, time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                              recall_score, average_precision_score)

plt.rcParams['figure.facecolor'] = '#0c1118'
plt.rcParams['axes.facecolor']   = '#131b24'
plt.rcParams['axes.edgecolor']   = '#1e2d3d'
plt.rcParams['text.color']       = '#e8edf2'
plt.rcParams['axes.labelcolor']  = '#e8edf2'
plt.rcParams['xtick.color']      = '#6b7a8d'
plt.rcParams['ytick.color']      = '#6b7a8d'
plt.rcParams['grid.color']       = '#1e2d3d'
print('✅ Libraries loaded')

## 1. Load Preprocessed Data

In [ ]:
X_train, y_train = joblib.load('models/train_resampled.pkl')
X_test,  y_test  = joblib.load('models/test_set.pkl')

print(f'Training : {X_train.shape}  — Fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'Test     : {X_test.shape}   — Fraud: {y_test.sum():,}  ({y_test.mean()*100:.4f}%)')

## 2. Define Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        C=0.01,
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,       # already balanced by SMOTE
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
}
print('Models defined:')
for name in models: print(f'  • {name}')

## 3. Train & Evaluate All Models

In [ ]:
results = {}

for name, model in models.items():
    print(f'\n🔧 Training {name}...')
    t0 = time.time()

    if name == 'XGBoost':
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)
    else:
        model.fit(X_train, y_train)

    elapsed = time.time() - t0
    y_pred  = model.predict(X_test)
    y_prob  = model.predict_proba(X_test)[:, 1]

    metrics = {
        'AUC-ROC'  : round(roc_auc_score(y_test, y_prob), 4),
        'Avg Prec' : round(average_precision_score(y_test, y_prob), 4),
        'F1'       : round(f1_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'Train (s)': round(elapsed, 1),
        'model'    : model,
        'y_prob'   : y_prob,
    }
    results[name] = metrics

    print(f'   ✅ Done in {elapsed:.1f}s')
    print(f'   AUC-ROC : {metrics["AUC-ROC"]}')
    print(f'   F1      : {metrics["F1"]}')
    print(f'   Recall  : {metrics["Recall"]}')

## 4. Comparison Table

In [ ]:
summary = pd.DataFrame({
    name: {k: v for k, v in m.items() if k not in ('model','y_prob')}
    for name, m in results.items()
}).T

print('\n📊 Model Comparison:')
print(summary.to_string())

# Highlight best in each column
best_model_name = summary['AUC-ROC'].idxmax()
print(f'\n🏆 Best model by AUC-ROC: {best_model_name}')

## 5. ROC Curve Comparison

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

colors = {'Logistic Regression': '#ffa502', 'Random Forest': '#0099ff', 'XGBoost': '#00d4aa'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC curves
for name, m in results.items():
    fpr, tpr, _ = roc_curve(y_test, m['y_prob'])
    axes[0].plot(fpr, tpr, color=colors[name],
                 label=f'{name} (AUC={m["AUC-ROC"]})', linewidth=2)
axes[0].plot([0,1],[0,1],'--', color='#6b7a8d', linewidth=1, label='Random baseline')
axes[0].set_title('ROC Curve', fontweight='bold', fontsize=13)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.02])

# Precision-Recall curves
for name, m in results.items():
    prec, rec, _ = precision_recall_curve(y_test, m['y_prob'])
    axes[1].plot(rec, prec, color=colors[name],
                 label=f'{name} (AP={m["Avg Prec"]})', linewidth=2)
axes[1].axhline(y_test.mean(), color='#6b7a8d', linestyle='--',
                linewidth=1, label=f'Baseline ({y_test.mean():.4f})')
axes[1].set_title('Precision-Recall Curve', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=9)
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('training_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: training_roc_pr_curves.png')

## 6. XGBoost Feature Importance

In [ ]:
xgb_model = results['XGBoost']['model']
feature_names = list(pd.read_json('models/feature_columns.json', typ='series'))

importances = pd.Series(xgb_model.feature_importances_, index=feature_names)
top15 = importances.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top15.index, top15.values, color='#00d4aa', edgecolor='none', alpha=0.85)
ax.set_title('XGBoost — Top 15 Feature Importances', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, top15.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, color='#e8edf2')
plt.tight_layout()
plt.savefig('training_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: training_feature_importance.png')

## 7. Save the Best Model

In [ ]:
best_model = results['XGBoost']['model']

joblib.dump(best_model, 'models/fraud_model.pkl')
print('✅ Model saved → models/fraud_model.pkl')

# Save model metadata
meta = {
    'model_type'   : 'XGBoost',
    'n_estimators' : 300,
    'auc_roc'      : results['XGBoost']['AUC-ROC'],
    'f1_score'     : results['XGBoost']['F1'],
    'precision'    : results['XGBoost']['Precision'],
    'recall'       : results['XGBoost']['Recall'],
    'trained_on'   : str(pd.Timestamp.now().date()),
    'train_samples': int(len(X_train)),
    'test_samples' : int(len(X_test)),
}
with open('models/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('✅ Metadata saved → models/model_metadata.json')
print()
print('All saved files:')
for f in os.listdir('models'): print(f'  models/{f}')
print()
print('➡️  Next: 04_Evaluation.ipynb')